# Nairobi OS Quickstart: The Sovereign API (v0.3.1)

Welcome to the **Heavy Iron**. This notebook demonstrates the v0.3.1 "Frictionless" API refit of Nairobi OS, building it from source, and verifying its hardware-accelerated capabilities.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KevinKenya/nairobi-connector-open-source/blob/main/quickstart.ipynb)

## 🏗️ Architecture: The Triad
- **Axum Refinery**: The Rust-powered engine for zero-copy ingestion.
- **Sovereign API**: Fluent Python bindings for high-performance analytics.
- **Lagos Visual Cortex**: Hardware-accelerated plotting directly from shared memory.

### 🛠️ Step 1: Environment Setup & Tooling

Nairobi OS requires a Linux environment with D-Bus and the Rust toolchain. The following cell prepares Google Colab for the Heavy Iron Forge.

In [1]:
import sys
import os
import subprocess
import shutil
import time
import glob

def log_step(msg):
    print(f"[FORENSIC LOG {time.strftime('%H:%M:%S')}] {msg}")

if 'google.colab' in sys.modules:
    log_step("Initializing Managed Colab Environment...")

    # 1. System Dependencies
    log_step("Installing system dependencies (D-Bus, Build tools)...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    deps = ["dbus-x11", "build-essential", "pkg-config", "libssl-dev", "libdbus-1-dev", "clang", "cmake"]
    subprocess.run(["apt-get", "install", "-y", "-qq"] + deps, check=True)

    # 2. Rust Toolchain
    if shutil.which("cargo") is None:
        log_step("Bootstrapping Rust toolchain...")
        subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
        os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.cargo/bin")

    # 3. Python Tooling
    log_step("Installing Maturin and AnyWidget...")
    subprocess.run(["pip", "install", "maturin", "anywidget", "traitlets", "kagglehub"], check=True)

    # 4. Repository Acquisition
    repo_dir = "/content/nairobi-connector-open-source"
    if not os.path.exists(repo_dir):
        log_step("Cloning Nairobi OS repository...")
        subprocess.run(["git", "clone", "https://github.com/KevinKenya/nairobi-connector-open-source.git", repo_dir], check=True)
    os.chdir(repo_dir)
    log_step(f"Working directory: {os.getcwd()}")
else:
    log_step("Native environment detected. Ensure Rust and D-Bus are installed.")

[FORENSIC LOG 09:49:08] Initializing Managed Colab Environment...
[FORENSIC LOG 09:49:08] Installing system dependencies (D-Bus, Build tools)...
[FORENSIC LOG 09:49:40] Bootstrapping Rust toolchain...
[FORENSIC LOG 09:50:07] Installing Maturin and AnyWidget...
[FORENSIC LOG 09:50:15] Cloning Nairobi OS repository...
[FORENSIC LOG 09:50:16] Working directory: /content/nairobi-connector-open-source


### 🛠️ Step 2: The Forge (Building the v0.3.1 Wheel)

We build a `manylinux` compatible wheel to ensure broad Linux support. This compiles the Rust refinery, the Lagos daemon, and the Python bindings.

In [11]:
import re
log_step("--- COMMENCING HEAVY IRON FORGE ---")

# Patch build_wheel.sh to skip manylinux audit correctly
if os.path.exists("build_wheel.sh"):
    with open("build_wheel.sh", "r") as f:
        content = f.read()

    # Replace compatibility flags or incorrect skip-audit flags with the correct Maturin flag
    new_content = re.sub(r"--compatibility\s+['\"]?\w+['\"]?", "--skip-auditwheel", content)
    new_content = new_content.replace("--skip-audit ", "--skip-auditwheel ")
    new_content = new_content.replace("--skip-audit\n", "--skip-auditwheel\n")

    with open("build_wheel.sh", "w") as f:
        f.write(new_content)

subprocess.run(["chmod", "+x", "build_wheel.sh"], check=True)

# Execute the patched script
process = subprocess.Popen(["/bin/bash", "./build_wheel.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    wheels = glob.glob("target/wheels/*.whl")
    if wheels:
        latest_wheel = max(wheels, key=os.path.getctime)
        log_step(f"Installing forged wheel: {latest_wheel}")
        subprocess.run(["pip", "install", "--force-reinstall", latest_wheel], check=True)
        log_step("✅ Nairobi OS v0.3.1 Installed.")
    else:
        log_step("❌ Error: Wheel not found in target/wheels!")
else:
    log_step(f"❌ Build failed with exit code {process.returncode}")

[FORENSIC LOG 10:21:05] --- COMMENCING HEAVY IRON FORGE ---
Nairobi OS v0.3.1: Heavy Iron Build Orchestrator
Step 1: Compiling Axum Refinery...
   --> crates/nairobi-axum-refinery/src/ingest.rs:102:49
    |
102 | pub async fn ingest(&mut self, file_path: &str, delimiter: &str, encoding: &str) -> ImperialResult<OwnedFd> {
    |                                                 ^^^^^^^^^ help: if this is intentional, prefix it with an underscore: `_delimiter`
    |
    = note: `#[warn(unused_variables)]` (part of `#[warn(unused)]`) on by default

   --> crates/nairobi-axum-refinery/src/ingest.rs:102:66
    |
102 | pub async fn ingest(&mut self, file_path: &str, delimiter: &str, encoding: &str) -> ImperialResult<OwnedFd> {
    |                                                                  ^^^^^^^^ help: if this is intentional, prefix it with an underscore: `_encoding`

    Finished `release` profile [optimized] target(s) in 0.36s
note: to see what the problems were, use the option `--fu

### 🔍 Step 3: Version Verification

Ensure we are running the correct version of the Heavy Iron.

In [12]:
import importlib
import importlib.metadata
import sys

# Refresh the import cache to recognize the newly installed wheel
importlib.invalidate_caches()

try:
    import nairobi_os
    version = importlib.metadata.version("nairobi-os")
    print(f"[VERIFICATION] Nairobi OS Version: {version}")
    if version != "0.3.1":
        print("⚠️ Warning: Version mismatch! Expected 0.3.1")
    else:
        print("✅ Module 'nairobi_os' successfully imported.")
except Exception as e:
    print(f"❌ Failed to verify version: {e}")

[VERIFICATION] Nairobi OS Version: 0.3.1
✅ Module 'nairobi_os' successfully imported.


### 📂 Step 4: Dataset Acquisition

We import the Historical NBA dataset via `kagglehub`.

In [ ]:
import kagglehub
path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')

print('Data source import complete.')
print(f'Path to dataset: {path}')

# Locate the Player Box Scores CSV
csv_files = glob.glob(os.path.join(path, "**/*player_box_scores.csv"), recursive=True)
if not csv_files:
    csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

csv_file = csv_files[0] if csv_files else None
print(f"Using CSV for analysis: {csv_file}")

### 🚀 Step 5: The Ignition

The `connect()` call prepares the D-Bus session and ignites the Axum Refinery daemon.

In [ ]:
nairobi_os.connect()
print("✅ Refinery is live.")

### ⚙️ Step 6: Low-Level API Strike

Direct interaction with the `_core` bindings for maximum transparency.

In [ ]:
import json

log_step("Low-Level Ingestion...")
handle_id = nairobi_os.data.ingest(csv_file)
print(f"Handle ID: {handle_id}")

log_step("Low-Level Crunching (PTS column)...")
try:
    stats_json = nairobi_os.data.crunch(handle_id, "pts") # Case sensitivity depends on CSV header
    stats = json.loads(stats_json)
    print(f"Points Statistics: {json.dumps(stats, indent=2)}")
except Exception as e:
    print(f"⚠️ Crunch failed (check column names): {e}")

### 💎 Step 7: High-Level Sovereign API

Fluent, Pandas-like operations powered by Rust's speed.

In [ ]:
df = nairobi_os.read_csv(csv_file)
print(f"Sovereign Frame Handle: {df.handle_id}")

try:
    # Fluent column access
    mean_pts = df.pts.mean()
    max_pts = df.pts.max()
    print(f"Average Points: {mean_pts:.2f}")
    print(f"Maximum Points: {max_pts:.2f}")

    # SQL Distillation
    log_step("Executing Relational SQL Strike...")
    high_scorers = df.query("SELECT pts FROM dataset WHERE pts > 30")
    print(f"High Scorers Frame: {high_scorers.handle_id}")
except Exception as e:
    print(f"⚠️ Sovereign API error: {e}")

### 👁️ Step 8: Lagos Visual Cortex & Forensic Verification

Hardware-accelerated plotting directly from `memfd`. We'll also perform a forensic check on the daemon's health.

In [ ]:
log_step("Spawning Lagos Vision Widget...")
try:
    # Render the distilled data
    widget = df.pts.calculate() # This just shows we can crunch it
    # For actual plotting:
    plot = df.plot(width=800, height=400)
    display(plot)
except Exception as e:
    print(f"❌ Lagos Plotting Failed: {e}")

log_step("--- LAGOS FORENSIC VERIFICATION ---")
if os.path.exists("/tmp/lagos.log"):
    with open("/tmp/lagos.log", "r") as f:
        logs = f.read()
        if "[LAGOS_PORT:" in logs:
            print("✅ Lagos Daemon: Ignition Success (Port discovered in logs)")
        else:
            print("⚠️ Lagos Daemon: Port signature not found in /tmp/lagos.log")
        if "wgpu" in logs.lower():
            print("✅ Lagos Daemon: Hardware Acceleration (wgpu) Initialized")
else:
    print("❌ Lagos Daemon: Forensic log (/tmp/lagos.log) missing!")

### 🛑 Step 9: Shutdown

Cleanly terminate the refinery daemons.

In [ ]:
nairobi_os.stop_refinery()
log_step("Refinery Decommissioned. Mission Complete.")